## import libraries

In [1]:
import ee
from dotenv import load_dotenv
import os
import ee
import pandas as pd
from pathlib import Path
import pickle

## initialize google earth engine

In [2]:
ee.Authenticate()

True

In [3]:
load_dotenv()  
project = os.getenv("PROJECT")
ee.Initialize(project=project)

## read datasets

In [4]:
root_embeddings = Path("../_data/exports/embeddings_exports")
root_dataset = Path("../_data/exports/crop_country_exports")

In [23]:
def build_files_dataframe(embeddings, dataset):
    rows = []

    folders = [dataset, embeddings]

    for folder in folders:
        for file in folder.glob("*"):
    
            if file.suffix not in [".csv", ".parquet"]:
                continue
            name = file.stem  
            parts = name.split("_")

            if "embedding" in parts:
                year = parts[-2]
                country = parts[-3]
                crop = "_".join(parts[:-3])
                file_type = "embedding"
            else:
                year = parts[-1]
                country = parts[-2]
                crop = "_".join(parts[:-2])
                file_type = "dataset"
            
            rows.append({
                "path": str(file),
                "crop": crop,
                "country": country,
                "year": int(year),
                "type": file_type
            })

    return pd.DataFrame(rows)

In [123]:
files = build_files_dataframe(root_embeddings, root_dataset)
files.head(3)

,path,crop,country,year,type
0,../_data/exports/crop_country_exports/maize_co...,maize_corn_popcorn,pt,2018,dataset
1,../_data/exports/crop_country_exports/maize_co...,maize_corn_popcorn,pt,2019,dataset
2,../_data/exports/crop_country_exports/potatoes...,potatoes,bg,2018,dataset


In [124]:
# 2018, 2019, 2020, 2021, 2022 já está
country = "fr"
year = [2023] # 2023]  

In [125]:
files = files[(files['country'] == country) & (files['year'].isin(year))]
files

,path,crop,country,year,type
35,../_data/exports/crop_country_exports/maize_co...,maize_corn_popcorn,fr,2023,dataset
67,../_data/exports/crop_country_exports/potatoes...,potatoes,fr,2023,dataset
82,../_data/exports/crop_country_exports/winter_b...,winter_barley,fr,2023,dataset
105,../_data/exports/embeddings_exports/potatoes_f...,potatoes,fr,2023,embedding
111,../_data/exports/embeddings_exports/winter_bar...,winter_barley,fr,2023,embedding
126,../_data/exports/embeddings_exports/maize_corn...,maize_corn_popcorn,fr,2023,embedding


In [126]:
# pairing embeddings with datasets
df_datasets = files[files['type'] == 'dataset'].drop(columns=['type'])
df_embeddings = files[files['type'] == 'embedding'].drop(columns=['type'])

pairs = pd.merge(df_datasets, 
                 df_embeddings, 
                 on=['crop', 'country', 'year'], 
                 suffixes=('_dataset', '_embedding'))

pairs

,path_dataset,crop,country,year,path_embedding
0,../_data/exports/crop_country_exports/maize_co...,maize_corn_popcorn,fr,2023,../_data/exports/embeddings_exports/maize_corn...
1,../_data/exports/crop_country_exports/potatoes...,potatoes,fr,2023,../_data/exports/embeddings_exports/potatoes_f...
2,../_data/exports/crop_country_exports/winter_b...,winter_barley,fr,2023,../_data/exports/embeddings_exports/winter_bar...


In [127]:
# # for many years
# pairs = pairs[pairs['year'] == year]
# pairs.head(3)

# for a single year
pairs = pairs[pairs['year'] == year[0]]
pairs.head()

,path_dataset,crop,country,year,path_embedding
0,../_data/exports/crop_country_exports/maize_co...,maize_corn_popcorn,fr,2023,../_data/exports/embeddings_exports/maize_corn...
1,../_data/exports/crop_country_exports/potatoes...,potatoes,fr,2023,../_data/exports/embeddings_exports/potatoes_f...
2,../_data/exports/crop_country_exports/winter_b...,winter_barley,fr,2023,../_data/exports/embeddings_exports/winter_bar...


In [128]:
datasets_dict = {}
embeddings_dict = {}

for _, row in pairs.iterrows():
    key = f"{row['crop']}_{row['country']}_{row['year']}"
    
    datasets_dict[key] = pd.read_csv(row['path_dataset'])
    embeddings_dict[key] = pd.read_parquet(row['path_embedding'])

print(f"Successfully loaded {len(datasets_dict)} pairs.")
print(f"Dictionaries keys are  {datasets_dict.keys()} pairs.")

Successfully loaded 3 pairs.
Dictionaries keys are  dict_keys(['maize_corn_popcorn_fr_2023', 'potatoes_fr_2023', 'winter_barley_fr_2023']) pairs.


In [129]:
datasets_dict['potatoes_fr_2023'].head(2)

,country_id,year,hcat4_code,hcat4_name,long,lat
0,fr,2023,3310300000,potatoes,2.496135,49.920616
1,fr,2023,3310300000,potatoes,2.691521,44.263970


In [130]:
embeddings_dict['potatoes_fr_2023'].head(2)

,tile_lon,tile_lat,pixel_row,pixel_col,crs,embedding,long_lat,crop,country_id,year
0,2.45,49.95,887,691,EPSG:32631,"[1.2859496, -1.4028541, 0.17535676, 5.3776073,...","[2.496135461542439, 49.920616394376154]",potatoes,fr,2023
1,2.65,44.25,403,733,EPSG:32631,"[1.4465419, -3.3752644, 0.5424532, 1.6273596, ...","[2.691520853649659, 44.263970183921906]",potatoes,fr,2023


## get point images

| Data Type   | Access Method                     | Description                                                |
|--------------|----------------------------------|------------------------------------------------------------|
| Dataset     | `datasets_dict[key].iloc[i]`     | The i-th row of tabular data (labels, etc.).              |
| Embeddings   | `embeddings_dict[key].iloc[i]`   | The i-th row of the embedding vector.                     |
| Satellite    | `gee_dict[key][i]['s2']`         | The full Sentinel-2 time series for that exact point.     |

In [131]:
def get_sits_data(lat, lon, year):
    point = ee.Geometry.Point([lon, lat])
    start = f'{year}-01-01'
    end = f'{year}-12-31'

    s1_bands = ['VV', 'VH']
    
    s1_col = (ee.ImageCollection("COPERNICUS/S1_GRD")
              .filterBounds(point)
              .filterDate(start, end)
              .filter(ee.Filter.eq('instrumentMode', 'IW'))
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
              .select(s1_bands))
    
    s2_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
    s2_col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
              .filterBounds(point)
              .filterDate(start, end)
              #.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 100))
              .select(s2_bands))
    
    s1_raw = s1_col.getRegion(point, 10).getInfo() # 10 is the resolution
    s2_raw = s2_col.getRegion(point, 10).getInfo() 


    if len(s1_raw) <= 1:
        print(f"No S1 data found for ({lat}, {lon}) in {year}")
        s1_raw = []

    return s1_raw, s2_raw

In [132]:
def to_df(raw_data):
    df = pd.DataFrame(raw_data[1:], columns=raw_data[0])
    df['date'] = pd.to_datetime(df['time'], unit='ms')
    return df.drop(columns=['id', 'time'])

In [133]:
datasets_dict.keys()

dict_keys(['maize_corn_popcorn_fr_2023', 'potatoes_fr_2023', 'winter_barley_fr_2023'])

In [134]:
images_dict = {}

for key in datasets_dict.keys(): # todos os datasets
    df_main = datasets_dict[key]
    
    file_results = []
    
    print(f"Starting extraction for {key} ({len(df_main)} points)...")
    this_year = datasets_dict[key]['year'].iloc[0   ]
    for idx, row in df_main.iterrows():
        lat, lon = row['lat'], row['long']
        
        try:
            s1_raw, s2_raw = get_sits_data(lat, lon, this_year)
            df_s1 = to_df(s1_raw)
            df_s2 = to_df(s2_raw)

            file_results.append({'s1': df_s1, 
                                 's2': df_s2, 
                                 'point_coord': (lon, lat)})
            
        except Exception as e:
            print(f"Error at row {idx} in {key}: {e}")
            file_results.append(None) 
            
    images_dict[key] = file_results

Starting extraction for maize_corn_popcorn_fr_2023 (500 points)...
Starting extraction for potatoes_fr_2023 (500 points)...
Starting extraction for winter_barley_fr_2023 (500 points)...


In [135]:
images_dict.keys()

dict_keys(['maize_corn_popcorn_fr_2023', 'potatoes_fr_2023', 'winter_barley_fr_2023'])

## export results

In [136]:
images_output_dir = Path("../_data/exports/image_points_exports")
images_output_dir.mkdir(parents=True, exist_ok=True)

def save_gee_data(data_dict, folder):
    for key, content in data_dict.items():
        file_path = folder / f"{key}_image.pkl"
        with open(file_path, 'wb') as f:
            pickle.dump(content, f)
    print(f"Successfully saved {len(data_dict)} files to {folder}")

save_gee_data(images_dict, images_output_dir)

Successfully saved 3 files to ../_data/exports/image_points_exports


In [ ]:
# # to load later
# with open('_data/exports/image_points_exports/potatoes_dk_2019_image.pkl', 'rb') as f:
#     ooo = pickle.load(f)

# ooo[0]['point_coord']